# MILK10k — Step 6b: Grad-CAM with faithfulness checks (GPU, ~20–30 min)
Uses the fine-tuned unimodal ConvNeXtV2 networks from notebook 02 (`net_{C,D}_fold{f}.pt`). Every lesion is explained by the network of **its own test fold** (the network never trained on it).
1. **Deletion test** — progressively blur the pixels Grad-CAM ranks most important and track the predicted-class probability. A faithful map makes the probability fall *faster* than a random smooth map (lower area under the curve).
2. **Centre focus** — share of Grad-CAM mass in the central 25% of the image (lesions are centred; random = 0.25), for correct vs incorrect predictions.
3. **Figures** — representative correct cases (one per key diagnosis, chosen at random, not cherry-picked) and melanomas the dermoscopy model called nevus.

**Inputs:** attach **notebook-o2** (networks + folds + OOF predictions) and **MILK10_DATASET** (images). Settings: **GPU T4**, **Internet On** (to rebuild the ConvNeXtV2 architecture).

In [ ]:
import os, numpy as np, pandas as pd, warnings, time
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms as T
from PIL import Image
from transformers import AutoModel
from scipy.stats import wilcoxon

def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
NET0 = find_file("net_D_fold0.pt"); FOLDS_CSV = find_file("milk10k_lesions_folds.csv")
assert NET0 and FOLDS_CSV, "attach notebook-o2 output (networks + folds). If the folder is empty, restart the session."
STAGE2 = os.path.dirname(NET0)
les = pd.read_csv(FOLDS_CSV)
JPG0 = find_file(les["isic_id_D"].iloc[0] + ".jpg"); assert JPG0, "attach MILK10_DATASET"
IMG_DIR = os.path.dirname(JPG0)
OUT = "/kaggle/working/stage6b"; os.makedirs(OUT, exist_ok=True)

CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
BACKBONE, IMG_SIZE = "facebook/convnextv2-tiny-22k-224", 224
N_PER_FOLD = 40                      # lesions per fold per modality for the deletion test (400 total)
STEPS = np.linspace(0, 1, 11)        # 0%, 10%, ..., 100% of pixels blurred
TRAPZ = getattr(np, "trapezoid", None) or np.trapz
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(device, "| networks:", STAGE2)
y = les["y"].values.astype(int); fold = les["fold"].values
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TF_EVAL = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])
def img_path(i, mod): return f"{IMG_DIR}/{les[f'isic_id_{mod}'].iloc[i]}.jpg"
def load_x(i, mod): return TF_EVAL(Image.open(img_path(i, mod)).convert("RGB")).to(device)
def load_rgb(i, mod): return np.asarray(Image.open(img_path(i, mod)).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))

## Network (identical to notebook 02) + Grad-CAM on the last ConvNeXt stage

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(BACKBONE)
        self.dim = self.backbone.config.hidden_sizes[-1]
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.dim, K))
    def forward(self, x): return self.head(self.backbone(pixel_values=x).pooler_output)

def load_net(mod, f):
    net = Net().to(device)
    net.load_state_dict(torch.load(f"{STAGE2}/net_{mod}_fold{f}.pt", map_location=device))
    return net.eval()

class GradCAM:
    def __init__(self, net):
        self.net, self.act, self.grad = net, None, None
        net.backbone.encoder.stages[-1].register_forward_hook(self._fwd)
    def _fwd(self, module, inp, out):
        o = out[0] if isinstance(out, (tuple, list)) else out
        self.act = o
        if o.requires_grad: o.register_hook(self._bwd)
    def _bwd(self, g): self.grad = g
    def __call__(self, x, target=None):
        self.net.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.net(x)
            if target is None: target = logits.argmax(1)
            logits.gather(1, target[:, None]).sum().backward()
        w = self.grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam / (cam.flatten(1).max(1)[0][:, None, None] + 1e-8)
        return cam.detach(), logits.detach().softmax(1), target

BLUR = T.GaussianBlur(51, sigma=20)
@torch.no_grad()
def deletion_curve(net, x, rank_map, target):
    """x: (3,H,W) normalised image; rank_map: (H,W). Blur pixels in decreasing rank order."""
    H, W = rank_map.shape; N = H * W
    blurred = BLUR(x[None])[0]; order = torch.argsort(rank_map.flatten(), descending=True)
    batch = []
    for p in STEPS:
        m = torch.zeros(N, device=x.device); m[order[:int(round(p * N))]] = 1; m = m.view(1, H, W)
        batch.append(x * (1 - m) + blurred * m)
    return net(torch.stack(batch)).softmax(1)[:, target].cpu().numpy()

def random_smooth_map(seed):
    g = torch.Generator().manual_seed(int(seed))
    r = torch.rand(1, 1, 7, 7, generator=g)          # same 7x7 resolution as Grad-CAM
    return F.interpolate(r, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].to(device)

yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
CENTRE = torch.tensor((np.abs(yy - IMG_SIZE / 2) < IMG_SIZE / 4) & (np.abs(xx - IMG_SIZE / 2) < IMG_SIZE / 4), device=device)
def centre_share(cam): return (cam[CENTRE].sum() / (cam.sum() + 1e-8)).item()

## 1–2. Deletion test and centre focus (400 lesions per modality, test-fold networks)

In [ ]:
rows, curves = [], {"C": {"gradcam": [], "random": []}, "D": {"gradcam": [], "random": []}}
t0 = time.time()
for mod in ["C", "D"]:
    for f in range(5):
        net = load_net(mod, f); gc = GradCAM(net)
        idx = les.index[fold == f].to_series().sample(N_PER_FOLD, random_state=100 + f).values
        for i in idx:
            x = load_x(i, mod)[None]
            cam, prob, tgt = gc(x); t = int(tgt.item())
            d_cam = deletion_curve(net, x[0], cam[0], t)
            d_rnd = deletion_curve(net, x[0], random_smooth_map(i), t)
            curves[mod]["gradcam"].append(d_cam); curves[mod]["random"].append(d_rnd)
            rows.append(dict(modality=mod, fold=f, lesion=i, true=CLASSES[y[i]], pred=CLASSES[t], correct=(t == y[i]),
                             p0=d_cam[0], auc_gradcam=TRAPZ(d_cam, STEPS), auc_random=TRAPZ(d_rnd, STEPS),
                             centre_share=centre_share(cam[0])))
        del net, gc; torch.cuda.empty_cache()
        print(f"{mod} fold {f} done ({time.time() - t0:.0f}s)")
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/deletion_results.csv", index=False)

summ = []
for mod, d in res.groupby("modality"):
    stat = wilcoxon(d["auc_gradcam"], d["auc_random"])
    summ.append(dict(modality={"C": "clinical", "D": "dermoscopy"}[mod], n=len(d),
                     deletion_AUC_gradcam=d["auc_gradcam"].mean(), deletion_AUC_random=d["auc_random"].mean(),
                     gradcam_better_share=(d["auc_gradcam"] < d["auc_random"]).mean(), wilcoxon_p=stat.pvalue,
                     centre_share_correct=d.loc[d.correct, "centre_share"].mean(),
                     centre_share_wrong=d.loc[~d.correct, "centre_share"].mean()))
summ = pd.DataFrame(summ).set_index("modality"); summ.to_csv(f"{OUT}/faithfulness_summary.csv")
print("Lower deletion AUC = probability falls faster = more faithful map. Centre share: random = 0.25")
summ.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, mod in zip(ax, ["C", "D"]):
    for kind, col in [("gradcam", "C3"), ("random", "gray")]:
        arr = np.array(curves[mod][kind]); m, se = arr.mean(0), arr.std(0) / np.sqrt(len(arr))
        a.plot(STEPS * 100, m, color=col, label=kind); a.fill_between(STEPS * 100, m - 1.96 * se, m + 1.96 * se, color=col, alpha=0.2)
    a.set_title({"C": "Clinical network", "D": "Dermoscopy network"}[mod]); a.set_xlabel("% of pixels blurred (most important first)")
    a.set_ylabel("probability of predicted class"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_deletion_curves.png", dpi=200); plt.show()

## 3. Representative cases (random correct case per diagnosis) and melanomas called nevus

In [ ]:
oof = {m: pd.read_csv(f"{STAGE2}/oof_{m}.csv").set_index("lesion_id").reindex(les["lesion_id"]) for m in "CD"}
predD = oof["D"][CLASSES].values.argmax(1); predC = oof["C"][CLASSES].values.argmax(1)
rng = np.random.RandomState(0)
good = []
for cls in ["BCC", "MEL", "SCCKA", "AKIEC", "VASC", "NV"]:
    cand = np.where((y == CLASSES.index(cls)) & (predD == y))[0]
    if len(cand): good.append(int(rng.choice(cand)))
fail = np.where((y == CLASSES.index("MEL")) & (predD == CLASSES.index("NV")))[0]
fail = [int(i) for i in rng.choice(fail, min(4, len(fail)), replace=False)]

def cams_for(indices):
    out = {}
    for f in sorted(set(fold[indices])):
        nets = {m: load_net(m, f) for m in "CD"}; gcs = {m: GradCAM(nets[m]) for m in "CD"}
        for i in [j for j in indices if fold[j] == f]:
            out[i] = {}
            for m in "CD":
                cam, prob, tgt = gcs[m](load_x(i, m)[None])
                out[i][m] = (cam[0].cpu().numpy(), CLASSES[int(tgt)], float(prob[0, tgt]))
        del nets, gcs; torch.cuda.empty_cache()
    return out

def panel(indices, cams, title, fname):
    fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3 * len(indices)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(indices):
        for c_, (m, lab) in enumerate([("C", "clinical"), ("D", "dermoscopy")]):
            rgb = load_rgb(i, m); cam, pl, pp = cams[i][m]
            axes[r, 2 * c_].imshow(rgb); axes[r, 2 * c_].set_title(f"{lab} | true {CLASSES[y[i]]}", fontsize=9)
            axes[r, 2 * c_ + 1].imshow(rgb); axes[r, 2 * c_ + 1].imshow(cam, cmap="jet", alpha=0.45)
            axes[r, 2 * c_ + 1].set_title(f"pred {pl} ({pp:.2f})", fontsize=9)
    for a in axes.ravel(): a.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=160); plt.show()

cams = cams_for(good + fail)
panel(good, cams, "Grad-CAM, randomly chosen correct cases (test-fold networks)", "fig_gradcam_correct.png")
panel(fail, cams, "Melanomas the dermoscopy network called nevus", "fig_gradcam_mel_as_nv.png")